In [1]:
from kaggle_secrets import UserSecretsClient

from datasets import load_dataset
from tqdm import tqdm
from datasets import load_dataset
from huggingface_hub import hf_hub_download

import os
import json, copy, re, itertools

In [2]:
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("hf_key")

In [3]:
dataset_ids = {
    "HuggingFaceFW/fineweb-edu": {"name": "sample-10BT", "streaming": True},
    "li2017dailydialog/daily_dialog": {"revision": "refs/convert/parquet"},
    "facebook/empathetic_dialogues": {"revision": "refs/convert/parquet"},
    "bavard/personachat_truecased": {"data_dir": "full", "revision": "refs/convert/parquet"},
    "google-research-datasets/mbpp": {"name": "sanitized"}                                  
}
all_datasets = dict()
for ds in dataset_ids:
    try:
        all_datasets[ds] = load_dataset(ds, **dataset_ids[ds])
        print(f"{ds} PROBLEM: OK")
    except Exception as problem:
        print(f"{ds} PROBLEM: {problem}")

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

HuggingFaceFW/fineweb-edu PROBLEM: OK


default/train/0000.parquet:   0%|          | 0.00/3.61M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/334k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/120 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/257 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/43 [00:00<?, ? examples/s]

Generating prompt split:   0%|          | 0/7 [00:00<?, ? examples/s]

google-research-datasets/mbpp PROBLEM: OK


In [4]:
for ds in all_datasets:
    print("=" * 15, f"DATASET {ds}", "=" * 15)
    current_dataset = all_datasets[ds]
    if dataset_ids[ds].get("streaming", False):
        dataset_example = current_dataset[next(iter(current_dataset))]
        print(f"N_SAMPLES: UNKNOWN")
        print(f"DATA EXAMPLE: {next(iter(dataset_example))}")
    else:
        dataset_example = current_dataset[tuple(current_dataset.keys())[0]]
        print("N_SAMPLES:")
        total_rows = 0
        for split_name in current_dataset:
            print(f"\t LEN {split_name}: {len(current_dataset[split_name])} rows")
            total_rows += len(current_dataset[split_name])
        print(f"WHOLE DATASET LEN: {total_rows}")
        
            
        print(f"DATA EXAMPLE: {dataset_example[0]}")

    print(f"SPLITS:{current_dataset.keys()}")
    print(f"COLUMNS: {dataset_example.features}", '\n')

=============== DATASET HuggingFaceFW/fineweb-edu ===============
N_SAMPLES: UNKNOWN
DATA EXAMPLE: {'text': 'The Independent Jane\nFor all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.\nElizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day. Her refusal of Mr. Darcy while triggered by anger showed a level of independence that left him shocked and stunned.\nThe freedom she exhibited in finally accepting him in direct defiance of Lady Catherine and knowing her father would disapprove was unusual even for Austen. In her last book Anne Elliot is persuaded to refuse Captain Wentworth at Lady Russel’s insistence.\nAlthough Jane played by the rules of the day, all of her writing is infused with how she wanted life to be. She ‘screams’ her outrage at the limitations for women in Emma.\nWhen accosted by Mrs. Elton, Jane Fairfa

In [5]:
for _ in all_datasets["bavard/personachat_truecased"]:
    for row in all_datasets["bavard/personachat_truecased"][_]:
        if row["candidates"][-1] == "Hi, how are you doing? I'm getting ready to do some cheetah chasing to stay in shape.":
            print(row)
            break

{'personality': ['I like canning and whittling.', 'To stay in shape, I chase cheetahs at the zoo.', 'In high school, I came in 6th in the 100 meter dash.', 'I eat exclusively meat.'], 'candidates': ['Do you walk her when it is cold?', 'Thank you, do you have cheese in your fridge? I love it.', 'Sounds funny. Do you have any hobbies?', 'We should. My boys would love that. Thank you.', 'How about Metallica I like them a lot.', "Yes. Did he make a dog movie? I've 3 who get free care.", 'Hi there, I am Bob.', 'That is a great idea. What do you do when you have down time.', "No, I'm thinking about grabbing something on my way to work. Do u work?", 'Well you much live in a cold city like new York city too.', 'Hey you like water? I do.', 'Oh that is cool. I like athletes.', 'Basketball is my thing. 5 mil a year makes it great!', 'Yeah he did the team really good. But I think we will have a good season.', 'I spend a lot on pens. I have a weird obsession with them.', 'I like ethnic foods I am t

In [6]:
is_empty = lambda text: text is None or text.strip() == ""

In [7]:
def adapter_MBPP(datasetdict):
    splits = datasetdict.keys()
    OUTPUT_TEMPLATE = {
        "source": "mbpp",
        "source_split": "",
        "source_id": "",
        "turns": [{"role": "user", "text": ""}, {"role": "bot", "text": ""}],
        "meta": {}
    }
    prompt_and_code_lost, prompt_lost, code_lost = 0, 0, 0
    for split in splits:
        for row in datasetdict[split]:
            if is_empty(row["prompt"]) and is_empty(row["code"]):
                prompt_and_code_lost += 1
                continue
            elif is_empty(row["prompt"]):
                prompt_lost += 1
                continue
            elif is_empty(row["code"]):
                code_lost += 1
                continue
            OUTPUT = copy.deepcopy(OUTPUT_TEMPLATE)       
            OUTPUT["source_split"] = split
            OUTPUT["source_id"] = str(row["task_id"])
            OUTPUT["turns"][0]["text"] = row["prompt"]
            OUTPUT["turns"][1]["text"] = "```python\n" + row["code"].rstrip() + "\n```"
            yield OUTPUT
    print(f"BOTH: {prompt_and_code_lost}, PROMPT: {prompt_lost}, CODE: {code_lost}")

In [8]:
def adapter_FineWeb(iterabledataset):
    OUTPUT_TEMPLATE = {
        "source": "fineweb-edu",
        "source_split": "train",
        "source_id": "",
        "text": "",
        "meta": {"score": "", "token_count": "", "url": "", "language_score": ""}
    }
    text_lost = 0
    for row in iterabledataset:
        if is_empty(row["text"]):
            text_lost += 1
            continue
        OUTPUT = copy.deepcopy(OUTPUT_TEMPLATE)
        OUTPUT["source_id"] = row["id"]
        OUTPUT["text"] = row["text"]
        OUTPUT["meta"]["score"] = row["score"]
        OUTPUT["meta"]["token_count"] = row["token_count"]
        OUTPUT["meta"]["url"] = row["url"]
        OUTPUT["meta"]["language_score"] = row["language_score"]
        yield OUTPUT
    print(f"LOST TEXTS: {text_lost}")

In [9]:
def detokenize_dd(text):
    patterns = {
        "spaces_left_edge": [re.compile(r"^\s*"), ""],
        "spaces_right_edge": [re.compile(r"(?<=\S)\s*$"), ""],
        "space_before_punctuation": [re.compile(r"\s+(?=[,.?!;:])"), ""],
        "glue_torn_apostrophe": [re.compile(r"(?<=\w)\s'\s(?=\w)"), "'"],
        "insert_space_after_end_of_sentence": [re.compile(r"(?<=[a-z])([.?!])(?=[A-Z])"), r"\1 "],
        "repeated_spaces_into_one": [re.compile(r"\s{2,}"), " "]
    }
    for rule in patterns:
        pattern = patterns[rule]
        text = pattern[0].sub(pattern[1], text)
    return text

In [10]:
def adapter_DailyDialog(datasetdict):
    splits = datasetdict.keys()
    OUTPUT_TEMPLATE = {
        "source": "dailydialog",
        "source_split": "",
        "source_id": "",
        "turns": [],
        "meta": {"emotions": []}
    }
    lostdialog, less2phrases = 0, 0
    for split in splits:
        emotions_list = datasetdict[split].features["emotion"].feature.names
        for idx, row in enumerate(datasetdict[split]):
            OUTPUT = copy.deepcopy(OUTPUT_TEMPLATE)
            OUTPUT["source_split"] = split
            OUTPUT["source_id"] = str(idx)
            for i in range(len(row["dialog"])):
                temp = detokenize_dd(row["dialog"][i])
                if len(temp) == 0:
                    lostdialog += 1
                    break
                if i % 2 == 0:
                    OUTPUT["turns"].append({"role": "user", "text": temp})
                else:
                    OUTPUT["turns"].append({"role": "bot", "text": temp})
            else:
                if len(OUTPUT["turns"]) < 2:
                    less2phrases += 1
                    continue
                OUTPUT["meta"]["emotions"] = [emotions_list[value] for value in row["emotion"]]
                yield OUTPUT
    print(lostdialog, less2phrases)

In [11]:
def adapter_EmpatheticDialogues(datasetdict):
    splits = datasetdict.keys()
    OUTPUT_TEMPLATE = {
        "source": "empatheticdialogues",
        "source_split": "",
        "source_id": "",
        "turns": [],
        "meta": {"emotion": "", "situation": ""}
    }
    dialogues = dict()
    for split in splits:
        for row in datasetdict[split]:
            dialogues.setdefault(row["conv_id"], [[], set([])])[0].append(row)
            dialogues[row["conv_id"]][1].add(split)
    emptyphrase, less2phrase = 0, 0
    gnumeration, galteration = 0, 0
    split_on_different = 0
    for d in dialogues:
        flag1 = False
        act_d = dialogues[d]
        act_d[0].sort(key=lambda x: x["utterance_idx"])

        OUTPUT = copy.deepcopy(OUTPUT_TEMPLATE)
        d_splits = list(act_d[1])
        if len(d_splits) != 1:
            split_on_different += 1
            continue
        OUTPUT["source_split"] = d_splits[0]
        OUTPUT["source_id"] = d
        alterationmask, alteration, numeration = -1, 0, 0
        for idx, phrase in enumerate(act_d[0]):
            if idx + 1 != phrase["utterance_idx"]: numeration += 1

            if idx != 0 and alterationmask == phrase["speaker_idx"]:
                alteration += 1
            alterationmask = phrase["speaker_idx"]

            role_for_data = "user" if idx % 2 == 0 else "bot"
            text_for_data = phrase["utterance"].replace("_comma_", ",")
            if is_empty(text_for_data):
                emptyphrase += 1
                flag1 = True
                break
            OUTPUT["turns"].append(
                {"role": role_for_data,
                 "text": text_for_data}
            )
        if flag1:
            continue
        if len(OUTPUT["turns"]) < 2:
            less2phrase += 1
            continue
        gnumeration += 1 if numeration else 0
        galteration += 1 if alteration else 0

        if numeration: continue
        elif alteration: continue
        OUTPUT["meta"]["emotion"] = act_d[0][0]["context"]
        OUTPUT["meta"]["situation"] = act_d[0][0]["prompt"].replace("_comma_", ",")
        yield OUTPUT
    print(f"emptyphrase {emptyphrase}, less2phrase {less2phrase}, gnumeration {gnumeration}, \n galteration {galteration}, split_on_different {split_on_different}")

In [12]:
def adapter_PersonaChat(datasetdict):
    splits = datasetdict.keys()
    OUTPUT_TEMPLATE = {
        "source": "personachat",
        "source_split": "",
        "source_id": "",
        "turns": [],
        "meta": {"persona_user": [], "persona_bot": [], "source_id_pair": ""}
    }

    last_rows = dict()
    for split in splits:
        for row in datasetdict[split]:
            key = (split, row["conv_id"])
            if key not in last_rows or last_rows[key]["utterance_idx"] < row["utterance_idx"]:
                last_rows[key] = row

    sides = dict()
    collisions = 0
    for key, row in last_rows.items():
        history = list(row["history"])
        is_first = len(history) > 0 and history[0].startswith("__ SILENCE __")
        if is_first:
            history = history[1:]
        phrases = tuple(history + [row["candidates"][-1]])
        if phrases in sides:
            collisions += 1
            continue
        sides[phrases] = {"key": key, "persona": row["personality"], "is_first": is_first}

    paired = set()
    split_leakage, same_first, empty_phrase = 0, 0, 0
    for phrases, long_side in sides.items():
        short_side = sides.get(phrases[:-1])
        if short_side is None:
            continue
        paired.add(long_side["key"])
        paired.add(short_side["key"])

        if long_side["key"][0] != short_side["key"][0]:
            split_leakage += 1
            continue
        if long_side["is_first"] == short_side["is_first"]:
            same_first += 1
            continue
        if any(is_empty(p) for p in phrases):
            empty_phrase += 1
            continue

        if long_side["is_first"]:
            user_side, bot_side = long_side, short_side
        else:
            user_side, bot_side = short_side, long_side

        OUTPUT = copy.deepcopy(OUTPUT_TEMPLATE)
        OUTPUT["source_split"] = long_side["key"][0]
        OUTPUT["source_id"] = str(long_side["key"][1])
        for idx, phrase in enumerate(phrases):
            role = "user" if idx % 2 == 0 else "bot"
            OUTPUT["turns"].append({"role": role, "text": phrase})
        OUTPUT["meta"]["persona_user"] = list(user_side["persona"])
        OUTPUT["meta"]["persona_bot"] = list(bot_side["persona"])
        OUTPUT["meta"]["source_id_pair"] = str(short_side["key"][1])
        yield OUTPUT

    unpaired = len(last_rows) - len(paired)
    print(f"sides {len(last_rows)}, collisions {collisions}, unpaired {unpaired}, "
          f"split_leakage {split_leakage}, same_first {same_first}, empty_phrase {empty_phrase}")

In [13]:
def adapter_LLMDistillated(path):
    OUTPUT_TEMPLATE = {
        "source": "llmdistillated",
        "source_split": "train",
        "source_id": "",
        "turns": [],
        "meta": {"topic": "", "category": ""}
    }
    broken_json, missing_fields, less2phrase = 0, 0, 0
    empty_phrase, bad_roles = 0, 0
    with open(path, "r", encoding="UTF-8") as llmdistillated_data:
        for idx, line in enumerate(llmdistillated_data):
            line = line.strip()
            if not line:
                continue
            try:
                raw = json.loads(line)
            except json.JSONDecodeError:
                broken_json += 1
                continue
            if not isinstance(raw, dict) or not all(k in raw for k in ("turns", "topic", "category")):
                missing_fields += 1
                continue

            turns = raw["turns"]
            if not isinstance(turns, list) or len(turns) < 2:
                less2phrase += 1
                continue

            flag_empty, flag_roles = False, False
            for i, phrase in enumerate(turns):
                if not isinstance(phrase, dict) or "role" not in phrase or "text" not in phrase:
                    flag_roles = True
                    break
                expected_role = "user" if i % 2 == 0 else "bot"
                if phrase["role"] != expected_role:
                    flag_roles = True
                    break
                if is_empty(phrase["text"]):
                    flag_empty = True
                    break
            if flag_roles:
                bad_roles += 1
                continue
            if flag_empty:
                empty_phrase += 1
                continue

            OUTPUT = copy.deepcopy(OUTPUT_TEMPLATE)
            OUTPUT["source_id"] = str(idx)
            OUTPUT["turns"] = [{"role": p["role"], "text": p["text"]} for p in turns]
            OUTPUT["meta"]["topic"] = raw["topic"]
            OUTPUT["meta"]["category"] = raw["category"]
            yield OUTPUT
    print(f"broken_json {broken_json}, missing_fields {missing_fields}, less2phrase {less2phrase}, "
          f"empty_phrase {empty_phrase}, bad_roles {bad_roles}")

In [14]:
def write_jsonl(records, path):
    with open(path, "w", encoding="UTF-8") as out:
        c = 0
        for obj in records:
            out.write(json.dumps(obj, ensure_ascii=False) + "\n")
            c += 1
    return c

In [15]:
write_jsonl(
    itertools.chain(
        adapter_MBPP(all_datasets['google-research-datasets/mbpp']),
        adapter_DailyDialog(all_datasets['li2017dailydialog/daily_dialog']),
        adapter_EmpatheticDialogues(all_datasets['facebook/empathetic_dialogues']),
        adapter_PersonaChat(all_datasets['bavard/personachat_truecased']),
        adapter_LLMDistillated("/kaggle/input/datasets/absolute1000/llmdistillated/llmdistillated.jsonl")
    ),
    "/kaggle/working/SFT.jsonl"
)

BOTH: 0, PROMPT: 0, CODE: 0
0 0
emptyphrase 0, less2phrase 71, gnumeration 37, 
 galteration 3, split_on_different 0
sides 18878, collisions 0, unpaired 1328, split_leakage 0, same_first 0, empty_phrase 0
broken_json 1, missing_fields 0, less2phrase 0, empty_phrase 0, bad_roles 0


46669

In [16]:
data_files = []
for i in range(0, 3):
    data_files.append(hf_hub_download(
        repo_id="HuggingFaceFW/fineweb-edu",
        filename=f"sample/10BT/00{i}_00000.parquet",
        repo_type="dataset",
    ))
write_jsonl(adapter_FineWeb(load_dataset("parquet", data_files=data_files, split="train", streaming=True)), "/kaggle/working/PRETRAIN.jsonl")

sample/10BT/000_00000.parquet:   0%|          | 0.00/2.15G [00:00<?, ?B/s]

sample/10BT/001_00000.parquet:   0%|          | 0.00/2.15G [00:00<?, ?B/s]

sample/10BT/002_00000.parquet:   0%|          | 0.00/2.15G [00:00<?, ?B/s]

LOST TEXTS: 0


2182000